In [1]:
import joblib, os, json
from datetime import datetime
import mlflow

In [ ]:

def save_model_bundle(pipeline, model_name, feat_name, feat_cols,
                      team_features, player_features,
                      feature_version, model_save_dir,
                      metrics: dict):
    """
    Saves a self-describing model bundle as a .pkl file.
    The bundle contains the pipeline AND all metadata needed to
    reproduce predictions — no need to look anything up externally.
    
    Returns the path the file was saved to.
    """
    safe_feat = feat_name.replace(' ', '_').replace('(', '').replace(')', '').replace('+', '')
    filename  = f"{model_name}_{safe_feat}_{feature_version}.pkl"
    path      = os.path.join(model_save_dir, filename)

    bundle = {
        "pipeline":        pipeline,          # the actual sklearn/xgb pipeline
        "feat_cols":       feat_cols,          # exact ordered list of columns to pass in
        "team_features":   team_features,      # raw TEAM_FEATURES config
        "player_features": player_features,    # raw PLAYER_FEATURES config
        "feature_set":     feat_name,          # e.g. "All Features (Type 1+2+3)"
        "feature_version": feature_version,    # e.g. "v1", "v2"
        "model_name":      model_name,
        "metrics":         metrics,            # {"train": 0.68, "val": 0.44, "test": 0.45}
        "saved_at":        datetime.now().isoformat(),
    }

    joblib.dump(bundle, path)
    return path

In [ ]:
def load_model_bundle(path, fallback_feat_cols=None, fallback_team=None, fallback_player=None):
    """
    Loads a saved model — handles both formats:
      - New format: bundle dict saved by save_model_bundle()
      - Old format: bare pipeline saved by joblib.dump(pipeline, path)

    For old-format pkls, pass the fallback_* args so the caller knows
    which features to use. New-format pkls ignore the fallbacks entirely.
    """
    obj = joblib.load(path)

    if isinstance(obj, dict) and "pipeline" in obj:
        # ── New bundle format ─────────────────────────────────────────────────
        print(f"  [bundle] {obj['model_name']}  {obj.get('feature_version','?')}  "
              f"| {len(obj['feat_cols'])} features | metrics={obj.get('metrics',{})}")
        return obj["pipeline"], obj["feat_cols"], obj["team_features"], obj["player_features"]

    else:
        # ── Old bare-pipeline format ──────────────────────────────────────────
        print(f"  [bare pipeline] {path} — using fallback feature lists")
        if fallback_feat_cols is None:
            raise ValueError(
                "This pkl is a bare pipeline (saved before the bundle format was introduced). "
                "Pass fallback_feat_cols, fallback_team, and fallback_player so the caller "
                "knows which features to build the dataframe with."
            )
        return obj, fallback_feat_cols, fallback_team, fallback_player

In [ ]:


def compare_versions(experiment_name="football_prediction"):
    """
    Pulls all MLflow runs and prints a comparison table sorted by test accuracy.
    Use this after running v2 experiments to see if new features helped.
    """
    mlflow.set_tracking_uri("file:///C:/mlflow_tracking")
    runs = mlflow.search_runs(
        experiment_names=[experiment_name],
        order_by=["metrics.test_acc DESC"]
    )

    cols = ["tags.mlflow.runName", "tags.feature_version",
            "metrics.test_acc", "metrics.val_acc",
            "metrics.train_acc", "metrics.gap",
            "params.n_features"]

    available = [c for c in cols if c in runs.columns]
    summary   = runs[available].rename(columns={
        "tags.mlflow.runName":    "run",
        "tags.feature_version":   "version",
        "metrics.test_acc":       "test_acc",
        "metrics.val_acc":        "val_acc",
        "metrics.train_acc":      "train_acc",
        "metrics.gap":            "gap",
        "params.n_features":      "n_features",
    })

    print(summary.to_string(index=False))
    return summary